# GARCH Family Models - Kaggle Pipeline
Install necessary libraries.

In [ ]:
!pip install arch


In [ ]:
# Clone Github Repository (Nếu cần truy cập các file khác của dự án)
!git clone -b kaggle-implementation --single-branch https://github.com/nhanbayern/1003_EPA-Project_UIT.git
!ls 1003_EPA-Project_UIT


## 1. Dataset & Data Processing

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

DEFAULT_SEQ_LEN = 60
DEFAULT_VOL_WINDOW = 60

# Fixed split counts per dataset (Train, Val, Test) - data range 2010-2025
FIXED_SPLITS = {
    "VN30_INDEX": (1971, 1096, 925),
    "VN_INDEX": (1964, 1103, 925),
    "DAX_40": (1983, 1104, 972),
    "EURONEXT_100": (2005, 1115, 979),
    "IBEX_35": (1815, 1362, 923),
    "KOSPI_INDEX": (1944, 1109, 881),
    "SMI": (1987, 1135, 901),
    "SNP500": (1988, 1109, 927),
    "NIKKEI_225": (1677, 1174, 1062),
}

def _normalize_dataset_name(name: str) -> str:
    """Normalize dataset name to FIXED_SPLITS key (e.g., VN30_INDEX.csv -> VN30_INDEX)."""
    return str(name).upper().replace(".CSV", "")

def get_default_dataset_dir():
    kaggle_path = Path("/kaggle/input/datasets/trnhngv/historical-price/dataset")
    if kaggle_path.exists():
        return kaggle_path
    
    # Fallback to local path for testing (safe for Jupyter)
    try:
        return Path(__file__).resolve().parents[3] / "dataset"
    except NameError:
        return Path.cwd() / "dataset"

def load_close_series(csv_path):
    """Load close prices from CSV into pandas Series."""
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"Dataset file not found: {csv_path}")

    df = pd.read_csv(csv_path)
    df.columns = [col.strip().lower() for col in df.columns]

    if "close" not in df.columns:
        raise ValueError(f"Missing close column in {csv_path}")

    close = pd.to_numeric(df["close"], errors="coerce").dropna()

    if "time" in df.columns:
        idx = pd.to_datetime(df["time"], errors="coerce")
    elif "date" in df.columns:
        idx = pd.to_datetime(df["date"], errors="coerce")
    else:
        idx = pd.RangeIndex(len(close))

    valid = pd.Series(close.values, index=idx).dropna()
    if valid.empty:
        raise ValueError(f"No valid close prices in {csv_path}")

    return valid.sort_index()

def prepare_series(close_prices, volatility_window=DEFAULT_VOL_WINDOW):
    """
    Compute log-returns and 60-day rolling volatility from close prices.
    Returns: ln(Close_t / Close_{t-1}) * 100
    Vol: sqrt(mean((r_t - mean(r))^2)) over past 60 returns * 100
    """
    close = pd.Series(close_prices).dropna().astype(float)
    if close.empty:
        raise ValueError("close_prices is empty")

    returns = np.log(close / close.shift(1)).dropna()
    volatility = returns.rolling(window=int(volatility_window)).std(ddof=0).dropna()
    returns = returns.loc[volatility.index]

    returns = returns * 100.0
    volatility = volatility * 100.0

    return returns, volatility

def get_split_indices(n_samples, dataset_name):
    norm_name = _normalize_dataset_name(dataset_name)
    if norm_name not in FIXED_SPLITS:
        raise ValueError(f"Dataset '{dataset_name}' not in FIXED_SPLITS")

    train_cnt, val_cnt, test_cnt = FIXED_SPLITS[norm_name]
    return train_cnt, train_cnt + val_cnt

def create_sliding_windows(returns, volatility, seq_len=DEFAULT_SEQ_LEN):
    """
    Create sliding windows for training (target horizon=1).
    GARCH-LSTM is trained with h=1, and recursively predicts during testing.
    """
    r = pd.Series(returns).astype(float)
    v = pd.Series(volatility).astype(float)
    n = min(len(r), len(v))

    horizon = 1
    max_start = n - seq_len - horizon + 1

    windows = {
        "encoder_returns": [],
        "decoder_volatility": [],
        "target_returns": [],
        "target_variance": [],
    }

    for i in range(max_start):
        windows["encoder_returns"].append(r.iloc[i : i + seq_len].values)
        windows["decoder_volatility"].append(v.iloc[i : i + seq_len].values)
        windows["target_returns"].append(float(r.iloc[i + seq_len + horizon - 1]))
        windows["target_variance"].append(float(v.iloc[i + seq_len + horizon - 1]))

    return {k: np.asarray(v, dtype=np.float32) for k, v in windows.items()}

class GARCHDataset(Dataset):
    def __init__(self, data_dict):
        self.encoder_returns = torch.tensor(data_dict["encoder_returns"], dtype=torch.float32)
        self.decoder_volatility = torch.tensor(data_dict["decoder_volatility"], dtype=torch.float32)
        self.target_returns = torch.tensor(data_dict["target_returns"], dtype=torch.float32)
        self.target_variance = torch.tensor(data_dict["target_variance"], dtype=torch.float32)

    def __len__(self):
        return len(self.target_returns)

    def __getitem__(self, idx):
        return (
            self.encoder_returns[idx],
            self.decoder_volatility[idx],
            self.target_returns[idx],
            self.target_variance[idx]
        )

def get_dataloaders(csv_path, batch_size=32, seq_len=DEFAULT_SEQ_LEN):
    csv_path = Path(csv_path)
    dataset_name = csv_path.stem
    
    close_series = load_close_series(csv_path)
    returns, volatility = prepare_series(close_series)
    
    # Using fixed splits
    train_end, val_end = get_split_indices(len(returns), dataset_name)
    
    train_r, train_v = returns.iloc[:train_end], volatility.iloc[:train_end]
    val_r, val_v = returns.iloc[train_end:val_end], volatility.iloc[train_end:val_end]
    # For testing, we need sequential evaluation, so we might just use the raw series in testing logic,
    # but we can provide a test_loader for consistent batching if needed.
    test_r, test_v = returns.iloc[val_end-seq_len:], volatility.iloc[val_end-seq_len:]
    
    train_data = create_sliding_windows(train_r, train_v, seq_len)
    val_data = create_sliding_windows(val_r, val_v, seq_len)
    
    train_loader = DataLoader(GARCHDataset(train_data), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(GARCHDataset(val_data), batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, test_r, test_v, returns.index[val_end:]



## 2. Losses

In [ ]:
import torch
import torch.nn as nn

class TLoss(nn.Module):
    def __init__(self, v=5.0):
        super(TLoss, self).__init__()
        self.v = v

    def forward(self, y_pred_var, y_true_returns):
        y_pred_var = torch.clamp(y_pred_var, min=1e-6)

        term1 = torch.log(y_pred_var) / 2.0
        term2 = ((self.v + 1) / 2.0) * torch.log(
            1 + (y_true_returns ** 2) / ((self.v - 2) * y_pred_var)
        )

        loss = term1 + term2
        return torch.mean(loss)



## 3. Statistical GARCH Models

In [ ]:
import warnings
import numpy as np

try:
    from arch import arch_model
except ImportError as exc:
    arch_model = None
    ARCH_IMPORT_ERROR = exc
else:
    ARCH_IMPORT_ERROR = None

MODEL_SPECS = {
    "GARCH": {"vol": "GARCH", "p": 1, "o": 0, "q": 1, "power": 2.0},
    "GJR-GARCH": {"vol": "GARCH", "p": 1, "o": 1, "q": 1, "power": 2.0},
    "FI-GARCH": {"vol": "FIGARCH", "p": 1, "o": 0, "q": 1, "power": 2.0},
}

def _check_arch_ready():
    if arch_model is None:
        raise ImportError(
            "arch package is required for statistical baselines. "
            "Install it with: pip install arch"
        ) from ARCH_IMPORT_ERROR

def _fit_arch_model(history, model_name="GARCH", dist="t"):
    _check_arch_ready()
    if model_name not in MODEL_SPECS:
        valid = ", ".join(MODEL_SPECS.keys())
        raise ValueError(f"Unsupported model_name={model_name}. Valid: {valid}")

    spec = MODEL_SPECS[model_name]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = arch_model(
            np.asarray(history, dtype=float),
            mean="Zero",
            vol=spec["vol"],
            p=spec["p"],
            o=spec["o"],
            q=spec["q"],
            power=spec["power"],
            dist=dist,
            rescale=False,
        )
        fit_result = model.fit(disp="off", show_warning=False)
    return fit_result, model

def evaluate_stat_model(train_returns, test_returns, model_name="GARCH", dist="t", horizons=[1, 3, 5, 10, 21]):
    """
    Rolling forecast for multiple horizons.
    Returns:
    - predictions: dict of {h: array_of_volatilities}
    - all_params: list of dicts containing the fitted parameters
    """
    history = list(np.asarray(train_returns, dtype=float))
    test_arr = np.asarray(test_returns, dtype=float)
    max_h = max(horizons)

    if len(history) < 30:
        raise ValueError("Need at least 30 training points for stable ARCH fitting")

    predictions = {h: [] for h in horizons}
    all_params = []
    
    # Fallback to empirical variance if fit fails
    fallback_var = float(np.var(history)) if len(history) > 1 else 1.0
    fallback_var = max(fallback_var, 1e-6)

    # To avoid data leakage, at time t, we predict t+1..t+h
    for obs in test_arr:
        try:
            fit_result, model = _fit_arch_model(history, model_name=model_name, dist=dist)
            forecast = fit_result.forecast(horizon=max_h, reindex=False)
            pred_vars = forecast.variance.values[-1, :] # shape (max_h,)
            all_params.append(dict(fit_result.params))
        except Exception as e:
            pred_vars = np.full(max_h, fallback_var)
            all_params.append({})
            
        pred_vars = np.maximum(pred_vars, 1e-6)
        
        for h in horizons:
            # Aggregate variance: sum of variances from step 1 to h, divided by h, then sqrt
            agg_var = np.mean(pred_vars[:h])
            vol = np.sqrt(agg_var)
            predictions[h].append(vol)
            
        # Update history with the TRUE observation
        history.append(float(obs))
        
    for h in horizons:
        predictions[h] = np.asarray(predictions[h][:len(test_arr)], dtype=float)

    return predictions, all_params



## 4. GARCH-LSTM Hybrid Model

In [ ]:
import torch
import torch.nn as nn
import numpy as np

class GARCH_LSTM_Cell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(GARCH_LSTM_Cell, self).__init__()
        self.hidden_size = hidden_size

        self.W_f = nn.Linear(input_size, hidden_size)
        self.W_i = nn.Linear(input_size, hidden_size)
        self.W_c = nn.Linear(input_size, hidden_size)

        self.omega = nn.Parameter(torch.rand(1))
        self.alpha = nn.Parameter(torch.rand(1))
        self.beta = nn.Parameter(torch.rand(1))
        self.gamma = nn.Parameter(torch.rand(1))

        self.w = nn.Parameter(torch.tensor(0.1))

    def forward(self, eps_t_minus_1, sigma2_t_minus_1, c_t_minus_1):
        inputs = torch.cat([eps_t_minus_1, sigma2_t_minus_1], dim=-1)

        f_t = torch.sigmoid(self.W_f(inputs))
        i_t = torch.sigmoid(self.W_i(inputs))
        c_tilde = torch.tanh(self.W_c(inputs))

        c_t = f_t * c_t_minus_1 + i_t * c_tilde

        i_t_minus_1 = (eps_t_minus_1 < 0).float()

        o_t = (
            self.omega
            + self.alpha * (eps_t_minus_1 ** 2)
            + self.gamma * (eps_t_minus_1 ** 2) * i_t_minus_1
            + self.beta * sigma2_t_minus_1
        )

        sigma2_t = o_t * (1 + self.w * torch.tanh(c_t))
        sigma2_t = torch.clamp(sigma2_t, min=1e-6)

        return sigma2_t, c_t

class GARCHLSTMHybrid(nn.Module):
    def __init__(self, hidden_size=16):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = GARCH_LSTM_Cell(input_size=2, hidden_size=hidden_size)

    def forward(self, encoder_returns, decoder_variance):
        if encoder_returns.dim() == 1:
            encoder_returns = encoder_returns.unsqueeze(0)
        if decoder_variance.dim() == 1:
            decoder_variance = decoder_variance.unsqueeze(0)

        batch_size, seq_len = encoder_returns.shape
        c_t = torch.zeros(batch_size, self.hidden_size, device=encoder_returns.device)

        sigma2_prev = decoder_variance[:, 0:1]
        sigma2_path = []

        for t in range(1, seq_len):
            eps_prev = encoder_returns[:, t - 1 : t]
            sigma2_t, c_t = self.cell(eps_prev, sigma2_prev, c_t)
            sigma2_scalar = sigma2_t.mean(dim=-1, keepdim=True)
            sigma2_path.append(sigma2_scalar)
            sigma2_prev = decoder_variance[:, t : t + 1]

        if sigma2_path:
            return torch.cat(sigma2_path, dim=1)
        return torch.zeros(batch_size, 0, device=encoder_returns.device)

    def forecast_multi_variance(self, encoder_returns, decoder_variance, max_horizon=21):
        if encoder_returns.dim() == 1:
            encoder_returns = encoder_returns.unsqueeze(0)
        if decoder_variance.dim() == 1:
            decoder_variance = decoder_variance.unsqueeze(0)

        batch_size, seq_len = encoder_returns.shape
        c_t = torch.zeros(batch_size, self.hidden_size, device=encoder_returns.device)

        sigma2_prev = decoder_variance[:, 0:1]

        # Process historical window
        for t in range(1, seq_len):
            eps_prev = encoder_returns[:, t - 1 : t]
            _, c_t = self.cell(eps_prev, sigma2_prev, c_t)
            sigma2_prev = decoder_variance[:, t : t + 1]

        # Forecast h steps
        eps_prev = encoder_returns[:, -1:]
        forecasts = []
        for h in range(max_horizon):
            sigma2_t, c_t = self.cell(eps_prev, sigma2_prev, c_t)
            forecasts.append(sigma2_t.mean(dim=-1, keepdim=True))
            
            # For next step: we don't have true return, expected return is 0
            eps_prev = torch.zeros_like(eps_prev) 
            sigma2_prev = sigma2_t

        return torch.cat(forecasts, dim=1) # shape: (batch_size, max_horizon)

def evaluate_garch_lstm(model, test_returns, test_variance, seq_len=60, horizons=[1, 3, 5, 10, 21], device="cpu"):
    device = torch.device(device)
    model = model.to(device)
    model.eval()
    
    history_r = list(np.asarray(test_returns[:seq_len], dtype=float))
    history_v = list(np.asarray(test_variance[:seq_len], dtype=float))
    
    test_r = np.asarray(test_returns[seq_len:], dtype=float)
    test_v = np.asarray(test_variance[seq_len:], dtype=float)
    
    max_h = max(horizons)
    predictions = {h: [] for h in horizons}
    
    with torch.no_grad():
        for i, (r_next, v_next) in enumerate(zip(test_r, test_v)):
            enc_r = torch.tensor(history_r[-seq_len:], dtype=torch.float32, device=device).unsqueeze(0)
            dec_v = torch.tensor(history_v[-seq_len:], dtype=torch.float32, device=device).unsqueeze(0)
            
            pred_var_t = model.forecast_multi_variance(enc_r, dec_v, max_horizon=max_h) # (1, max_h)
            pred_vars = pred_var_t[0].cpu().numpy()
            pred_vars = np.maximum(pred_vars, 1e-6)
            
            for h in horizons:
                agg_var = np.mean(pred_vars[:h])
                vol = np.sqrt(agg_var)
                predictions[h].append(vol)
                
            history_r.append(float(r_next))
            history_v.append(float(v_next))
            
    for h in horizons:
        predictions[h] = np.asarray(predictions[h], dtype=float)
        
    return predictions



## 5. Training Loop

In [ ]:
import torch
import copy
import numpy as np

def train_garch_lstm_hybrid(
    model,
    criterion,
    train_loader,
    val_loader,
    device="cpu",
    epochs=100,
    learning_rate=1e-2,
    lr_factor=0.5,
    lr_patience=5,
    early_stopping_patience=20,
    min_lr=1e-6,
):
    device = torch.device(device)
    model = model.to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=lr_factor,
        patience=lr_patience,
        min_lr=min_lr,
    )

    best_state = None
    best_val_loss = float("inf")
    wait = 0

    history = {"train_loss": [], "val_loss": [], "lr": []}

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for enc_r, dec_v, target_r, _ in train_loader:
            enc_r = enc_r.to(device)
            dec_v = dec_v.to(device)
            target_r = target_r.to(device)

            optimizer.zero_grad(set_to_none=True)
            
            # Predict only horizon=1 during training for GARCH-LSTM density estimation
            pred_var = model.forecast_multi_variance(enc_r, dec_v, max_horizon=1)
            pred_var = pred_var.squeeze(-1) # shape: (batch_size,)
            
            loss = criterion(pred_var, target_r)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_losses.append(float(loss.item()))

        train_loss = float(np.mean(train_losses)) if train_losses else float("inf")

        model.eval()
        val_losses = []
        with torch.no_grad():
            for enc_r, dec_v, target_r, _ in val_loader:
                enc_r = enc_r.to(device)
                dec_v = dec_v.to(device)
                target_r = target_r.to(device)

                pred_var = model.forecast_multi_variance(enc_r, dec_v, max_horizon=1)
                pred_var = pred_var.squeeze(-1)
                val_loss = criterion(pred_var, target_r)
                val_losses.append(float(val_loss.item()))

        val_loss = float(np.mean(val_losses)) if val_losses else train_loss
        scheduler.step(val_loss)

        current_lr = optimizer.param_groups[0]["lr"]
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["lr"].append(current_lr)

        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if wait >= early_stopping_patience:
            print(f"Early stopping at epoch {epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history



## 6. Execution Pipeline

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

# Local imports

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def save_predictions_csv(index_name, model_name, predictions_dict, test_time, test_r, test_v, out_dir):
    """
    Saves predictions in the format: time, log_return, horizon, true_volatility, predict_volatility
    predictions_dict: {horizon: array_of_predicted_volatilities}
    """
    records = []
    
    # We only have predictions for timestamps starting from seq_len (or depending on the model offset)
    # For simplicity, ensure predictions align with the end of the test series.
    # predictions_dict[h] should have length equal to the number of test steps evaluated.
    
    horizons = sorted(list(predictions_dict.keys()))
    
    for h in horizons:
        preds = predictions_dict[h]
        # Align with the end of the test data
        valid_len = len(preds)
        
        # In case of seq_len offset for LSTM, we align with the last valid_len elements
        align_time = test_time[-valid_len:]
        align_r = test_r[-valid_len:]
        align_v = test_v[-valid_len:]
        
        for i in range(valid_len):
            records.append({
                "time": align_time[i],
                "log_return": align_r.iloc[i] if isinstance(align_r, pd.Series) else align_r[i],
                "horizon": h,
                "true_volatility": align_v.iloc[i] if isinstance(align_v, pd.Series) else align_v[i],
                "predict_volatility": preds[i]
            })
            
    df = pd.DataFrame(records)
    out_path = Path(out_dir) / f"{index_name}_{model_name}_predictions.csv"
    df.to_csv(out_path, index=False)
    print(f"Saved predictions to {out_path}")

def plot_predictions(index_name, model_name, predictions_dict, test_time, test_v, out_dir, horizon=21):
    """Plot true vs predicted volatility for a specific horizon."""
    if horizon not in predictions_dict:
        return
    
    preds = predictions_dict[horizon]
    valid_len = len(preds)
    align_time = test_time[-valid_len:]
    align_v = test_v[-valid_len:]
    
    plt.figure(figsize=(12, 6))
    plt.plot(align_time, align_v, label='True Volatility', alpha=0.7)
    plt.plot(align_time, preds, label=f'Predicted Volatility (h={horizon})', alpha=0.7)
    plt.title(f'{index_name} - {model_name} Volatility Forecast (Horizon={horizon})')
    plt.xlabel('Date')
    plt.ylabel('Volatility (%)')
    plt.legend()
    plt.grid(True)
    
    out_path = Path(out_dir) / f"{index_name}_{model_name}_h{horizon}_plot.png"
    plt.savefig(out_path)
    plt.close()

def run_pipeline():
    dataset_dir = get_default_dataset_dir()
    csv_files = glob.glob(str(dataset_dir / "*.csv"))
    
    if not csv_files:
        print(f"No CSV files found in {dataset_dir}")
        return
        
    out_base = Path("results")
    pred_dir = out_base / "predictions"
    viz_dir = out_base / "visualizations"
    model_dir = out_base / "model_params"
    
    ensure_dir(pred_dir)
    ensure_dir(viz_dir)
    ensure_dir(model_dir)
    
    horizons = [1, 3, 5, 10, 21]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    
    for csv_file in csv_files:
        index_name = Path(csv_file).stem.upper()
        print(f"\n{'='*50}\nProcessing {index_name}\n{'='*50}")
        
        # 1. Load data
        train_loader, val_loader, test_r, test_v, test_time = get_dataloaders(csv_file, batch_size=32)
        
        # Re-fetch the raw train data for statistical models which need raw returns
        close_series = load_close_series(csv_file)
        returns, volatility = prepare_series(close_series)
        train_end, val_end = get_split_indices(len(returns), index_name)
        raw_train_r = returns.iloc[:train_end]
        raw_test_r = returns.iloc[train_end:] # Use val+test for rolling forecast history, but evaluate on test
        
        # We need the full history up to the test set for rolling forecast
        stat_train_r = returns.iloc[:val_end]
        stat_test_r = returns.iloc[val_end:]
        stat_test_time = returns.index[val_end:]
        stat_test_v = volatility.iloc[val_end:]
        
        # 2. Train and Evaluate Statistical Models
        for stat_model_name in MODEL_SPECS.keys():
            print(f"--- Running {stat_model_name} ---")
            predictions, params = evaluate_stat_model(
                stat_train_r, stat_test_r, 
                model_name=stat_model_name, 
                dist="t", 
                horizons=horizons
            )
            
            # Save predictions
            save_predictions_csv(index_name, stat_model_name, predictions, stat_test_time, stat_test_r, stat_test_v, pred_dir)
            plot_predictions(index_name, stat_model_name, predictions, stat_test_time, stat_test_v, viz_dir, horizon=21)
            
            # Save params
            param_df = pd.DataFrame(params)
            param_df.to_csv(model_dir / f"{index_name}_{stat_model_name}_params.csv", index=False)
            
        # 3. Train and Evaluate GARCH-LSTM Hybrid
        print(f"--- Running GARCH-LSTM Hybrid ---")
        model = GARCHLSTMHybrid(hidden_size=16)
        criterion = TLoss(v=5.0)
        
        model, history = train_garch_lstm_hybrid(
            model, criterion, train_loader, val_loader, 
            device=device, epochs=50 # Using 50 epochs for quicker Kaggle runs, can be adjusted
        )
        
        # Save model weights
        torch.save(model.state_dict(), model_dir / f"{index_name}_GARCH_LSTM_weights.pth")
        
        # Evaluate multi-horizon
        # test_r and test_v from get_dataloaders already include seq_len history before the test set
        lstm_predictions = evaluate_garch_lstm(
            model, test_r, test_v, 
            seq_len=60, horizons=horizons, device=device
        )
        
        save_predictions_csv(index_name, "GARCH-LSTM-Hybrid", lstm_predictions, test_time, test_r.iloc[60:], test_v.iloc[60:], pred_dir)
        plot_predictions(index_name, "GARCH-LSTM-Hybrid", lstm_predictions, test_time, test_v.iloc[60:], viz_dir, horizon=21)

if __name__ == "__main__":
    run_pipeline()

